# Day 10 — Code-acting agents: one `run_python` tool vs many JSON tools
**Theory:** [day-10-code-and-computer-use-agents.md](../theory/day-10-code-and-computer-use-agents.md) · ⏱ ~60 min · Level: medium

⚠️ The local `run_python` here is a **learning sandbox only** (subprocess + timeout). Production needs a
container or a managed sandbox (see the theory note).

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Make a bigger dataset (500 orders) as CSV

In [ ]:
import random, subprocess, tempfile, json, sqlite3
import pandas as pd
from course_utils import run_agent
rng = random.Random(1)
items = [("Wireless earbuds", 49), ("Running shoes", 89), ("Coffee grinder", 35), ("Laptop stand", 29), ("Smart watch", 199)]
rows = []
for i in range(500):
    item, price = rng.choice(items)
    rows.append({"order_id": f"X{i:04}", "month": f"2026-{rng.randint(1, 10):02}", "item": item, "amount": price,
                 "tier": rng.choice(["gold", "standard", "standard"]), "returned": rng.random() < (0.25 if price > 100 else 0.08)})
DATA = pathlib.Path("week-2/practice/scratch/orders_500.csv"); DATA.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(DATA, index=False)
pd.read_csv(DATA).head()

## 2. The code tool

In [ ]:
def run_python(code: str) -> str:
    """LEARNING ONLY: subprocess + timeout. Not a security boundary."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
    try:
        p = subprocess.run([sys.executable, f.name], capture_output=True, text=True, timeout=20, cwd=str(REPO))
        return (p.stdout + p.stderr)[-3000:] or "(no output; use print())"
    except subprocess.TimeoutExpired:
        return "Error: timed out after 20s"

PY_TOOL = {"name": "run_python", "description": f"Run a Python script and get its stdout. pandas is installed. "
           f"The orders dataset is at '{DATA}' (columns: order_id, month, item, amount, tier, returned). Always print() results.",
           "input_schema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}
print(run_python("print(2 + 2)"))

## 3. Ask analytics questions

In [ ]:
QUESTIONS = ["What is the return rate per item? Sort highest first.",
             "Which month had the highest revenue from gold customers, and how much?",
             "Is the return rate for items over $100 higher than for cheaper items? Give both numbers."]
code_rows = []
for q in QUESTIONS:
    c0 = tracker.total()
    res = run_agent([{"role": "user", "content": q}], [PY_TOOL], {"run_python": run_python}, effort="low", verbose=False)
    code_rows.append({"question": q[:50], "steps": res.steps, "cost $": round(tracker.total() - c0, 4), "answer": res.text[:160]})
pd.DataFrame(code_rows)

## 4. Same questions with JSON tools only
A "get_orders(page)" tool returns raw rows. Watch steps and tokens explode, because every row passes through the context.

In [ ]:
ALL = pd.read_csv(DATA)
def get_orders(page: int = 0) -> str:
    return ALL.iloc[page * 100:(page + 1) * 100].to_json(orient="records")
JSON_TOOL = {"name": "get_orders", "description": "Get 100 orders per page (pages 0-4) as JSON rows.",
             "input_schema": {"type": "object", "properties": {"page": {"type": "integer"}}, "required": ["page"]}}
c0, before = tracker.total(), len(tracker.calls)
res = run_agent([{"role": "user", "content": QUESTIONS[0]}], [JSON_TOOL], {"get_orders": get_orders},
                effort="low", verbose=False, max_steps=10)
print(f"JSON tools: steps={res.steps}  input tokens={sum(c['input'] for c in tracker.calls[before:]):,}  "
      f"cost=${tracker.total() - c0:.4f}\n{res.text[:300]}")

## 5. Hosted sandbox: Claude's code execution tool
Anthropic runs the sandbox, so nothing executes on your machine.

In [ ]:
r = client.messages.create(model=MODEL, max_tokens=8000, output_config={"effort": "low"},
                           tools=[{"type": "code_execution_20260521", "name": "code_execution"}],
                           messages=[{"role": "user", "content": "Simulate 10,000 rolls of two dice. "
                                      "Report the mean of the sum and the most common sum."}])
tracker.add(MODEL, r.usage, "code_execution tool")
for b in r.content:
    if b.type == "text":
        print(b.text)
    elif "code_execution" in b.type:
        print(f"[{b.type}]")

## ✏️ Exercises
1. Ask the code agent to save a chart to `week-2/practice/scratch/chart.png` and open it.
2. Try to make `run_python` do something dangerous (e.g. list your home directory). Why is subprocess not a sandbox?
3. Give the capstone an "ops analytics" mode: open SQLite read-only with `sqlite3.connect("file:...db?mode=ro", uri=True)`.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None